In [16]:
import re
import pandas as pd
from os import listdir
from os.path import isfile, join, exists
import tempfile
import csv

data_dict = {}

def fix_row(row):
    """Auto-quote fields containing commas unless already quoted"""
    return [f'"{field}"' if ',' in field and not (field.startswith('"') and field.endswith('"')) else field for field in row]

path = "../data"
files = [re.sub(r".csv", "", f) for f in listdir(path) if isfile(join(path, f))]

for file in files:
    full_path = f"{path}/{file}.csv"
    if not exists(full_path):
        print(f"File not found: {full_path}")

    with open(full_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        lines = list(reader)

    header = lines[0]
    fixed_rows = [header]

    for i, row in enumerate(lines[1:], start=2):  # start=2 to match line numbers (1-based with header)
        if len(row) != 12:
            # Try to auto-fix by quoting comma-containing fields
            row = fix_row(row)
        if len(row) == 12:
            fixed_rows.append(row)
        else:
            print(f"Skipping malformed line {i} in {file}: {row}")

    # Save to a temporary cleaned file
    with tempfile.NamedTemporaryFile(mode="w+", newline='', delete=False, suffix=".csv") as tmp:
        writer = csv.writer(tmp)
        writer.writerows(fixed_rows)
        tmp_path = tmp.name
        
    try:
        data_dict[file] = pd.read_csv(tmp_path)
    except Exception as e:
        print(f"Failed to read cleaned CSV for {file}: {e}")

Skipping malformed line 4704 in books: ['16914', "The Tolkien Fan's Medieval Reader", 'David E. Smith (Turgon of TheOneRing.net', ' one of the founding members of this Tolkien website)/Verlyn Flieger/Turgon (=David E. Smith)', '3.58', '1593600119', '9781593600112', 'eng', '400', '26', '4', '4/6/2004', 'Cold Spring Press']
Skipping malformed line 5879 in books: ['22128', 'Patriots (The Coming Collapse)', 'James Wesley', ' Rawles', '3.63', '156384155X', '9781563841552', 'eng', '342', '38', '4', '1/15/1999', 'Huntington House Publishers']
Skipping malformed line 8981 in books: ['34889', "Brown's Star Atlas: Showing All The Bright Stars With Full Instructions How To Find And Use Them For Navigational Purposes And Department Of Trade Examinations.", 'Brown', ' Son & Ferguson', '0.00', '0851742718', '9780851742717', 'eng', '49', '0', '0', '5/1/1977', 'Brown Son & Ferguson Ltd.']


In [ ]:
for key, value in data_dict.items():
    print(key)
    print(value)
    print("---------")